[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_03/16_repaso_prueba_3.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 16 — Repaso integrador de la Unidad 3

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 3**

Este notebook combina los tres temas de la unidad: una línea con carga
compleja, una guía rectangular y una antena corta. Úselo para comprobar sus
desarrollos a mano.

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Resolver una línea completa: reflexión, ROE e impedancia de entrada.
2. Proponer un transformador de cuarto de onda para una carga real.
3. Calcular el corte y la longitud de onda de guía del modo dominante.
4. Evaluar eficiencia y ganancia de una antena corta.

In [ ]:
# Preparación del entorno: funciona igual en Google Colab y en una copia local.
import sys
import urllib.request
from pathlib import Path

MODULOS = ["utilidades_notebook.py", "constantes_fisicas.py", "lineas_transmision.py", "guias_y_antenas.py"]
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)

# Se exige que estén *todos* los módulos, no solo la carpeta: así, si otro
# notebook ya creó `src/` en esta misma sesión, igual se descarga lo que falte.
raiz = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if all((p / "src" / modulo).exists() for modulo in MODULOS)),
    None,
)
if raiz is None:  # Google Colab: descargar los módulos del curso.
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo in MODULOS:
        if not (raiz / "src" / modulo).exists():
            urllib.request.urlretrieve(URL_SRC + modulo, raiz / "src" / modulo)
sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import VELOCIDAD_LUZ
from lineas_transmision import (
    coeficiente_reflexion,
    razon_onda_estacionaria,
    impedancia_entrada,
    impedancia_cuarto_de_onda,
    longitud_electrica,
)
from guias_y_antenas import (
    frecuencia_de_corte,
    longitud_onda_guia,
    se_propaga,
    resistencia_radiacion_dipolo_corto,
    eficiencia_antena,
    ganancia_antena,
    a_decibelios,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. Cómo abordar un problema de esta unidad

Casi todos los problemas de la Unidad 3 se resuelven con la misma receta.

**Si le hablan de una línea:**

1. Calcule $\Gamma$ con la carga y $Z_0$.
2. De ahí sale la ROE y el balance de potencia.
3. Si le piden la impedancia en otro punto, use la fórmula de $Z_{\text{in}}$
   con $\beta l = 2\pi (l/\lambda)$.
4. Si le piden adaptar y la carga es real, use $Z_t = \sqrt{Z_0 Z_L}$.

**Si le hablan de una guía:**

1. Calcule el corte del modo dominante: $f_c = c/(2a)$.
2. Compare con la frecuencia de trabajo. Si $f < f_c$, no hay propagación y se
   acabó el problema.
3. Si $f > f_c$, calcule $\lambda_g$, $u_p$ y $u_g$ con el mismo factor
   $\sqrt{1 - (f_c/f)^2}$.

**Si le hablan de una antena corta:**

1. $R_r = 80\pi^2 (dl/\lambda)^2$.
2. Eficiencia, y después ganancia $G = \xi D$.
3. Convierta a dBi si se lo piden.

El error más común en pruebas es olvidar normalizar: revise siempre si la
longitud está en metros o en longitudes de onda.

## 3. Ecuaciones que necesitará

**Línea de transmisión:**

$$
\Gamma = \frac{Z_L - Z_0}{Z_L + Z_0},
\qquad
\mathrm{ROE} = \frac{1+|\Gamma|}{1-|\Gamma|},
$$

$$
Z_{\text{in}} = Z_0\frac{Z_L + jZ_0\tan\beta l}{Z_0 + jZ_L\tan\beta l},
\qquad
Z_t = \sqrt{Z_0 Z_L}.
$$

**Guía rectangular, modo dominante:**

$$
f_{c,10} = \frac{c}{2a},
\qquad
\lambda_g = \frac{\lambda_0}{\sqrt{1 - (f_c/f)^{2}}}.
$$

**Antena corta:**

$$
R_r = 80\pi^{2}\left(\frac{dl}{\lambda}\right)^{2},
\qquad
\xi = \frac{R_r}{R_r + R_p},
\qquad
G = \xi D = \xi \cdot 1.5 .
$$

## 4. Qué debería reconocer al ver los resultados

**Una ROE cercana a 1.5 es buena.** En la práctica se acepta hasta 2. Por
encima de eso conviene adaptar.

**Si $\lambda_g$ le da menor que $\lambda_0$, hay un error.** Dentro de una
guía la longitud de onda siempre crece. Es una comprobación gratis.

**Si la eficiencia le da mayor que 1, hay un error.** La eficiencia es una
fracción: siempre está entre 0 y 1.

**Si $R + T \ne 1$ o si $P_i \ne P_r + P_L$, hay un error.** La conservación
de la energía es la mejor herramienta de autocorrección que tiene.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: línea con carga compleja ---
Z0 = 50.0                     # impedancia característica [ohm]
ZL = 75.0 + 25.0j             # impedancia de la carga [ohm]
longitud_sobre_lambda = 0.2   # longitud de la línea, en longitudes de onda
carga_real_a_adaptar = 75.0   # carga real para el transformador [ohm]

# --- Problema 2: guía y antena ---
ancho_guia = 22.86e-3       # dimensión mayor de la guía a [m]
frecuencia = 9.0e9          # frecuencia de operación [Hz]
dl_sobre_lambda = 0.04      # longitud del dipolo en longitudes de onda
resistencia_perdidas = 0.5  # resistencia de pérdidas [ohm]
directividad = 1.5          # directividad del dipolo hertziano

## 6. Implementación

### 6.1 Problema 1 — la línea

In [ ]:
Gamma = coeficiente_reflexion(ZL, Z0)
roe = razon_onda_estacionaria(Gamma)
beta_l = longitud_electrica(longitud_sobre_lambda)
Zin = impedancia_entrada(ZL, Z0, beta_l)

Z_transformador = impedancia_cuarto_de_onda(Z0, carga_real_a_adaptar)

### 6.2 Problema 2 — la guía y la antena

In [ ]:
# El modo dominante TE10 corresponde a m = 1, n = 0: solo depende del ancho.
corte_dominante = frecuencia_de_corte(ancho_guia, ancho_guia, 1, 0)
propaga = bool(se_propaga(frecuencia, corte_dominante))
lambda_libre = VELOCIDAD_LUZ / frecuencia
lambda_guia = float(longitud_onda_guia(frecuencia, corte_dominante))

R_radiacion = resistencia_radiacion_dipolo_corto(dl_sobre_lambda)
eficiencia = eficiencia_antena(R_radiacion, resistencia_perdidas)
ganancia = ganancia_antena(eficiencia, directividad)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Reflexión, parte real", "Re(Gamma)", Gamma.real, "-"),
        ("Reflexión, parte imaginaria", "Im(Gamma)", Gamma.imag, "-"),
        ("Módulo de la reflexión", "|Gamma|", abs(Gamma), "-"),
        ("Fase de la reflexión", "arg(Gamma)", np.rad2deg(np.angle(Gamma)), "grados"),
        ("Razón de onda estacionaria", "ROE", roe, "-"),
        ("Impedancia de entrada, real", "Re(Z_in)", Zin.real, "ohm"),
        ("Impedancia de entrada, imaginaria", "Im(Z_in)", Zin.imag, "ohm"),
        ("Transformador para la carga real", "Z_t", Z_transformador, "ohm"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Corte del modo dominante", "f_c10", corte_dominante / 1.0e9, "GHz"),
        ("Frecuencia de operación", "f", frecuencia / 1.0e9, "GHz"),
        ("Longitud de onda libre", "lambda_0", lambda_libre * 1000.0, "mm"),
        ("Longitud de onda en la guía", "lambda_g", lambda_guia * 1000.0, "mm"),
        ("Resistencia de radiación", "R_r", R_radiacion, "ohm"),
        ("Eficiencia", "xi", eficiencia, "-"),
        ("Ganancia", "G", ganancia, "-"),
        ("Ganancia en decibelios", "G", a_decibelios(ganancia), "dBi"),
    ]
)

In [ ]:
if propaga:
    print(f"lambda_g / lambda_0 = {lambda_guia / lambda_libre:.6f}  (debe ser mayor que 1)")
    assert lambda_guia > lambda_libre, "lambda_g debería ser mayor"
    print("Comprobación de la guía: correcta.")
else:
    print(f"La frecuencia ({frecuencia / 1.0e9:.3f} GHz) está por debajo del corte "
          f"({corte_dominante / 1.0e9:.3f} GHz).")
    print("El modo no se propaga, así que lambda_g no está definida y sale 'nan'.")
    print("No es un error del cálculo: es lo que significa estar bajo el corte.")

print(f"Eficiencia = {eficiencia:.6f}  (debe estar entre 0 y 1)")
assert 0.0 <= eficiencia <= 1.0, "la eficiencia se salió de rango"
print("Comprobación de la antena: correcta.")

## 8. Visualización

A la izquierda, cómo transforma la línea. A la derecha, cómo cambia
$\lambda_g$ con la frecuencia: cerca del corte se dispara.

In [ ]:
fig, (eje_linea, eje_guia) = plt.subplots(1, 2, figsize=(9.5, 4.0))

longitudes = np.linspace(0.0, 0.5, 600)
Zin_curva = impedancia_entrada(ZL, Z0, longitud_electrica(longitudes))
eje_linea.plot(longitudes, abs(Zin_curva))
eje_linea.axhline(Z0, color="black", linestyle=":", label=f"Z_0 = {Z0:.0f} ohm")
eje_linea.scatter([longitud_sobre_lambda], [abs(Zin)], color="black", zorder=5,
                  label="caso evaluado")
eje_linea.set_xlabel("Longitud de la línea, en longitudes de onda")
eje_linea.set_ylabel("|Z_in| (ohm)")
eje_linea.set_title("Transformación en la línea")
eje_linea.legend(fontsize=8)

frecuencias = np.linspace(corte_dominante * 1.001, 15.0e9, 500)
lambda_g_curva = longitud_onda_guia(frecuencias, corte_dominante)
eje_guia.plot(frecuencias / 1.0e9, lambda_g_curva * 1000.0, color="tab:orange")
eje_guia.axvline(corte_dominante / 1.0e9, color="black", linestyle="--",
                 label="frecuencia de corte")
if propaga:
    eje_guia.scatter([frecuencia / 1.0e9], [lambda_guia * 1000.0], color="black",
                     zorder=5, label="caso evaluado")
eje_guia.set_xlabel("f (GHz)")
eje_guia.set_ylabel("lambda_g (mm)")
eje_guia.set_ylim(0.0, 200.0)
eje_guia.set_title("Dispersión del modo TE10")
eje_guia.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**La ROE es de aproximadamente 1.77: aceptable.** Con $|\Gamma| \approx 0.28$
vuelve un 7.7 % de la potencia. En la práctica no haría falta adaptar.

**La impedancia de entrada da algo muy distinto de la carga.** Ése es el punto
de la Unidad 3: lo que usted mide depende de dónde mida.

**$\lambda_g$ se dispara cerca del corte.** En el gráfico de la derecha la
curva sube verticalmente al acercarse a 6.56 GHz. En el límite, la longitud de
onda de guía se vuelve infinita: la onda deja de avanzar. Ésa es otra forma de
entender qué significa la frecuencia de corte.

**La antena es muy ineficiente.** Con $dl/\lambda = 0.04$ resulta
$R_r = 1.26~\Omega$ frente a 0.5 $\Omega$ de pérdidas: la eficiencia queda en
72 %. Antenas más cortas empeoran rápidamente porque $R_r$ va con el cuadrado.

**Los `assert` pasan.** Las dos comprobaciones de consistencia
($\lambda_g > \lambda_0$ y eficiencia en $[0,1]$) son las mismas que usted
debería hacer mentalmente en una prueba.

## 10. Ejercicios para experimentar

            1. Ponga `ZL = 50.0`. ¿Cuánto valen $\Gamma$, la ROE y $Z_{\text{in}}$? ¿Qué
               forma toma la curva de la izquierda?
            2. Ponga `longitud_sobre_lambda = 0.25` con `ZL = 75.0`. Verifique
               $Z_{\text{in}} = Z_0^2/Z_L$.
            3. Baje `frecuencia` a `6.0e9`, por debajo del corte. ¿Qué pasa con
               $\lambda_g$? ¿Qué mensaje de error aparece y por qué tiene sentido?
            4. Suba `frecuencia` a `15.0e9`. ¿Cuánto se acerca $\lambda_g$ a $\lambda_0$?
               ¿Por qué a frecuencias altas la guía "estorba" menos?
            5. Duplique `dl_sobre_lambda`. ¿Cuánto sube la eficiencia? ¿Y la ganancia en
               dBi?
            6. Arme su propia hoja de fórmulas para la evaluación de la unidad y
               contrástela con la sección 3 de este notebook. ¿Le falta alguna?